In [2]:
#import the datasets from the torch vision library
from torchvision import datasets
#import the operating system functionalities
import os
#import the pytorch library
import torch
#import the transform set from the torchvision library
from torchvision import transforms

In [ ]:
#Set the path to store the evaluation and test data
data_path = R'............\DL PJ'
test_path = r"............\DL PJ\test"
#create the specified directory if it does not exist
os.makedirs(data_path, exist_ok = True)
#download the CIFAR10 data to the specified directories and transform them into tensors
tensor_cifar10 = datasets.CIFAR10(data_path, train=True, download=True, transform=transforms.ToTensor())
tensor_cifar10_val = datasets.CIFAR10(test_path, train = False, download=True, transform = transforms.ToTensor())

In [7]:
#create a stack of the data to have 3 rows for each color channel
imgs = torch.stack([img_t for img_t, label in tensor_cifar10], dim = 3)
imgs_val = torch.stack([img_tv for img_tv, label_v in tensor_cifar10_val], dim = 3)

#calculate the mean and standard deviation for all color channels in the training set
mean = imgs.view(3, -1).mean(dim = 1)
std = imgs.view(3, -1).std(dim = 1)

#use the mean and std to normalize the data and randomly crop some images for better generalization
transforms.Normalize(mean = mean, std = std)
transformed_cifar10 = datasets.CIFAR10(data_path, train=True, download=False, transform = transforms.Compose([transforms.ToTensor(),transforms.RandomCrop(32, padding=4), transforms.Normalize(mean=mean, std  = std)]))
transformed_cifar10_val = datasets.CIFAR10(test_path, train = False, download=False,  transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=mean, std  = std)]))

In [9]:
#create a map to reassign labels to select data including only airplane and birds
label_map = {1:0, 9:1}
class_names = ['truck', 'autombile']
#reassign any label within the list [1,9]
cifar2 = [(img, label_map[label]) for img, label in transformed_cifar10 if label in [1,9]]
cifar2_val = [(img_val, label_map[label]) for img_val, label in transformed_cifar10_val if label in [1,9]]


In [10]:

#make the pytorch neural network library available for use as 'nn'
import torch.nn as nn
#make the optimization algorithms of pytorch available for use as optim
import torch.optim as optim

#create the model using a stack of layers with activation functions in between
model = nn.Sequential(nn.Linear(3072, 1024), nn.Tanh(), nn.Linear(1024,512), nn.Tanh(), nn.Linear(512, 256), nn.Tanh(), nn.Linear(256, 2))
#create an instance of torch.optim to receive the parameters and the learning rate
optimizer = optim.SGD(model.parameters(), lr = 1e-2)
#create a loss function handler
loss_fn = nn.CrossEntropyLoss()
#create a data conveyor to shuffle and send the cifar2 data in batches to the modelel
train_loader = torch.utils.data.DataLoader(cifar2, batch_size = 64, shuffle = True)

In [ ]:

#Create the training loop
#For every pass of a batch from the train loader,
for epoch in range(100):
	#assign imgage data to variable imgs and label data to labels
	for imgs, labels in train_loader:
		#set the batch size to number of samples loaded by trainloader
		batch_size = imgs.shape[0]
		#reshape the data to have single row of long features
		image_tensor = imgs.view(batch_size,-1)
		#convert the labels to pytorch tensors
		label_tensor = torch.tensor(labels)
		#forward pass the image data to the model and assign the prediction to 'out'
		out = model(image_tensor)
		#deduce the loss between the predicted and the actual values
		loss = loss_fn(out, label_tensor)
		#clear accumulated gradients
		optimizer.zero_grad()
		#calculate the loss contributing strenght of each weight
		loss.backward()
		#tweak the weights with gradients from the backpropagation
		optimizer.step()
	#print the epoch number and the loss calculated in that pass
	print("epoch number: {} loss: {}".format(epoch, loss))


In [ ]:
#create a data conveyor to shuffle and send the evaluation data in batches to the modelel
val_loader = torch.utils.data.DataLoader(cifar2_val, batch_size=6, shuffle=True)
#set containers for the count of correct and total validations
correct = 0
total = 0
#put the model into evaluation mode
model.eval()
#turn of gradient tracking
with torch.no_grad():
    # assign the image data to imgs and label data to labels
    for imgs, labels in val_loader:
        #set the batch size to the number of samples
        batch_size = imgs.shape[0]
        #set the number number of features to a long list of numbers
        imgs = imgs.view(batch_size, -1)
        #pass the imgs data to the model
        outputs = model(imgs)
        #find the highest probability and change it to 1, assign the prob. to 'value'
        value, predicted = torch.max(outputs, dim = 1)
        #for every pass, find the number the of samples and add it to the variale 'correct'
        total += labels.shape[0]
        #create a boolean of correct predictions and add the ones
        correct+= int((predicted == labels).sum())
#diveid the boolean sum by the number of samples to get an accuracy
print("Accuracy: {}".format(correct/total))